####This notebook will be used to read and update the Framework audit tables according to their use case.


In [0]:
#Setting up adls connection based on specific environment
environment = spark.conf.get("spark.databricks.clusterUsageTags.managedResourceGroup").lower()

clientIdKey="adpAceDataEdlClientId"
tenantIdKey="kellyTenantId"
clientSecret="adpAceDataEdlClientSecret"
keyVaultName="adp-kv-scope"

if ('tst-' in environment):
    storageName="kellyedldev"
    
elif ('sqa-' in environment):
    storageName="kellyedlsqa"
    
elif ('prd-' in environment):
    storageName="kellyedl"
    

In [0]:
clientId = dbutils.secrets.get(scope = keyVaultName, key = clientIdKey)
directoryId = dbutils.secrets.get(scope = keyVaultName, key = tenantIdKey)
secret = dbutils.secrets.get(scope = keyVaultName, key = clientSecret)

In [0]:
spark.conf.set("fs.azure.account.auth.type."+storageName+".dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type."+storageName+".dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id."+storageName+".dfs.core.windows.net",clientId)
spark.conf.set("fs.azure.account.oauth2.client.secret."+storageName+".dfs.core.windows.net",secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint."+storageName+".dfs.core.windows.net", "https://login.microsoftonline.com/"+directoryId+"/oauth2/token")

baseAdlsPath = "abfss://edl@"+storageName+".dfs.core.windows.net"
baseRawPath = baseAdlsPath + "/raw/aceData/"
baseCuratedPath = baseAdlsPath + "/curated/aceData/"

####1.Reading of audit table's

In [0]:
%sql
select * from aceMetastore.jobControl

In [0]:
%sql
select * from aceMetastore.pipelineAudit

In [0]:
%sql
select * from aceMetastore.dataQualityResult

In [0]:
%sql
select * from aceMetastore.adhocRuntable

####2.Updation of lastRunDate of jobControl table.


In [0]:
#We can assign any date in newDate variable and can use it to update the lastRunDate of jobControl table for specific sourceSystem

newDate='1900-01-01 00:00:00'
sourceSystem='bullhorndtp'

query=f"""UPDATE aceMetastore.jobControl
SET lastrundate = '{newDate}' 
where lower(sourceSystem)='{sourceSystem}'"""

display(spark.sql(query))

####3.Updation of adhocRunTable

In [0]:
#adhocRunTable is created for adhoc run of few tables which has high data volume and need optimization . So user has to run below update query to update the adhocRunFlag to 'y' for a particular sourceSystem and sourceObjectName.

adhocRunFlag='y'
sourceObjectName='dbo.JobOrder'
sourceSystem='bullhorndtp'

query=f"""UPDATE aceMetastore.adhocRuntable
SET adhocRunFlag = '{adhocRunFlag}' 
where lower(sourceSystem)='{sourceSystem}' and sourceObjectName= '{sourceObjectName}'"""

display(spark.sql(query))

####4.Reading of dataQualityResult table based on conditions

In [0]:
#Filtering only failed records in dataQualityResult table

query=f""" select * from aceMetastore.dataQualityResult where lower(dqStatus)="failure" """

display(spark.sql(query))

In [0]:
#Filtering only failed records in dataQualityResult table for current date 
query=f""" select * from aceMetastore.dataQualityResult where lower(dqStatus)="failure" and  DATE(dqEndTs) = CURDATE()"""

display(spark.sql(query))

####5.Reading of pipelineAudit table based on conditions

In [0]:
#Filtering of pipelineAudit table based on specific sourceSystem and sourceObjectName

sourceSystem='bullhorndtp'
sourceObjectName='dbo.category'

query=f""" select * from aceMetastore.pipelineAudit where sourceSystem='{sourceSystem}' and sourceObjectName = '{sourceObjectName}' """
display(spark.sql(query))

In [0]:
#Filtering of pipelineAudit table on current date for sourceToRaw job

query=f""" select * from aceMetastore.pipelineAudit where lower(stageName)="raw" and  DATE(stageEndTs) = CURDATE()"""

display(spark.sql(query))